# Step 7 · adv1e5: adversarial training + white-box re-attack (multilingual-E5 + para)
1. `scripts/adv_train.py`: trains BASE and ADV (with white-box rewrites of train seeds) for 3 seeds, compares them at
   1% / 5% val-FPR thresholds on test negatives, hard negatives, test positives, black-box attacks and held-out white-box
   rewrites (~30 min). Saves the seed-42 ADV checkpoint.
2. `scripts/attack_whitebox.py`: reruns the white-box attack against that ADV checkpoint, then the agent check (~45 min).

**Before you run:** push the latest commit · GPU **T4 x2** · Internet **On** · Secrets **Hugging Face** + **mcp-git** ticked ·
**Save Version → Save & Run All**. When done: Output → Download All → zip into the Project folder.


In [ ]:
# ---- settings for this run ----
REPO = "github.com/Arn15/mcp-poison-detect"
BRANCH = "main"
ADV_ARGS = "--tag adv1e5 --set detector.backbone=intfloat/multilingual-e5-base"
WB_TAG = "wb2_adve5"
WB_EXTRA = " --set detector.backbone=intfloat/multilingual-e5-base"
GPU_DEPS = True
MIN_COMMIT_FILE = "scripts/adv_train.py"


In [ ]:
import os, subprocess, base64
from kaggle_secrets import UserSecretsClient
sec = UserSecretsClient()
def secret(k):
    try: return sec.get_secret(k)
    except Exception: return None
HF_SECRET, GH_SECRET = "Hugging Face", "mcp-git"   # names of your Kaggle secrets
os.environ["HF_TOKEN"] = secret(HF_SECRET) or ""
gh = secret(GH_SECRET)
assert gh, "Secret 'mcp-git' not attached to this notebook (Add-ons -> Secrets)"
WORK = "/kaggle/working"

# Token goes in an HTTP header for this command only: never in the URL, .git/config, or error output.
b64 = base64.b64encode(f"x-access-token:{gh}".encode()).decode()
auth = ["-c", f"http.extraHeader=Authorization: Basic {b64}"]
def git(*args, cwd=None):
    r = subprocess.run(["git", *auth, *args], cwd=cwd, capture_output=True, text=True)
    if r.returncode != 0:
        msg = (r.stderr or r.stdout).replace(gh, "***")
        raise RuntimeError(f"git {args[0]} failed: {msg.strip()}")
    return r.stdout

os.chdir(WORK)
repo_dir = os.path.join(WORK, "mcp-poison-detect")
if os.path.isdir(os.path.join(repo_dir, ".git")):
    git("fetch", "-q", cwd=repo_dir); git("checkout", "-q", BRANCH, cwd=repo_dir); git("pull", "-q", cwd=repo_dir)
else:
    git("clone", "-q", "-b", BRANCH, f"https://{REPO}.git", repo_dir)
os.chdir(repo_dir)
print("commit:", git("log", "-1", "--oneline", cwd=repo_dir))
assert os.path.exists(MIN_COMMIT_FILE), f"{MIN_COMMIT_FILE} missing: push commit 058d739 first"


In [ ]:
!pip install -q -r requirements.txt
if GPU_DEPS:
    !pip install -q -r requirements-gpu.txt
!pip install -q -e . --no-deps


In [ ]:
import torch
!nvidia-smi --query-gpu=name,memory.total --format=csv
assert torch.cuda.device_count() >= 2, "Need GPU T4 x2"
!python scripts/adv_train.py {ADV_ARGS}


In [ ]:
import glob, os
adv_run = sorted(glob.glob("/kaggle/working/outputs/adv_train/*"))[-1]
ckpt = os.path.join(adv_run, "ckpt_adv")
assert os.path.exists(os.path.join(ckpt, "detector.json")), ckpt
print(open(os.path.join(adv_run, "adv_results.md")).read())
!python scripts/attack_whitebox.py --tag {WB_TAG} --set whitebox.detector_ckpt={ckpt}


In [ ]:
import glob, json
run = sorted(glob.glob("/kaggle/working/outputs/attack_whitebox/*"))[-1]
print(run)
print(json.dumps(json.load(open(f"{run}/summary.json")), indent=2))
